# Notebook 04 — Mô hình Dự báo Doanh thu
**Phụ trách:** Thành viên 4 (ML Engineer)

**Mục tiêu:** Dự báo Revenue từ 01/01/2023 – 01/07/2024

**Pipeline:**
1. Feature Engineering
2. TimeSeriesSplit Cross-Validation
3. Train LightGBM
4. SHAP Explanation
5. Tạo submission.csv


In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from utils import (
    load_all_data, parse_dates, set_seed,
    add_calendar_features, add_lag_features, add_rolling_features
)

SEED = 42
set_seed(SEED)

## 1. Load dữ liệu Train

In [ ]:
dfs = parse_dates(load_all_data())
sales = dfs['sales'].sort_values('Date').reset_index(drop=True)
web_traffic = dfs['web_traffic']
inventory   = dfs['inventory']
promotions  = dfs['promotions']

print(f'Train: {sales["Date"].min()} → {sales["Date"].max()}')
print(f'Shape: {sales.shape}')
sales.head()

## 2. Feature Engineering

In [ ]:
df = sales.copy()

# Calendar features
df = add_calendar_features(df, date_col='Date')

# Lag features (chỉ dùng quá khứ — không leak)
df = add_lag_features(df, target_col='Revenue', lags=[1, 7, 14, 30])
df = add_lag_features(df, target_col='COGS', lags=[1, 7, 14])

# Rolling features
df = add_rolling_features(df, target_col='Revenue', windows=[7, 14, 30])

# TODO: Thêm features từ web_traffic (sessions, bounce_rate...)
# TODO: Thêm features từ inventory (avg stockout_days, sell_through_rate...)
# TODO: Thêm promo flag (ngày có khuyến mãi đang chạy)

df.dropna(inplace=True)
print(f'After feature engineering: {df.shape}')

## 3. Train / Validation Split

In [ ]:
FEATURE_COLS = [c for c in df.columns if c not in ['Date', 'Revenue', 'COGS']]
TARGET = 'Revenue'

X = df[FEATURE_COLS]
y = df[TARGET]

print(f'Features: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

## 4. TimeSeriesSplit Cross-Validation

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = {'mae': [], 'rmse': [], 'r2': []}

lgb_params = {
    'objective': 'regression',
    'metric': 'mae',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'seed': SEED,
    'verbose': -1,
}

models = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

    model = lgb.train(
        lgb_params, train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )
    models.append(model)

    preds = model.predict(X_val)
    mae  = mean_absolute_error(y_val, preds)
    rmse = mean_squared_error(y_val, preds, squared=False)
    r2   = r2_score(y_val, preds)

    cv_scores['mae'].append(mae)
    cv_scores['rmse'].append(rmse)
    cv_scores['r2'].append(r2)
    print(f'Fold {fold+1} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}')

print(f'\nCV Mean MAE:  {np.mean(cv_scores["mae"]):.2f}')
print(f'CV Mean RMSE: {np.mean(cv_scores["rmse"]):.2f}')
print(f'CV Mean R2:   {np.mean(cv_scores["r2"]):.4f}')

## 5. SHAP Explanation

In [ ]:
# Dùng model fold cuối để giải thích
final_model = models[-1]
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X, plot_type='bar', max_display=15,
                  show=False)
plt.title('Top 15 Features — SHAP Importance')
plt.tight_layout()
plt.savefig('../report/figures/shap_importance.png', dpi=150)
plt.show()

## 6. Retrain trên toàn bộ train set & Tạo submission

In [ ]:
# Retrain full
full_data  = lgb.Dataset(X, label=y)
best_iter  = max(m.best_iteration for m in models)

final_full_model = lgb.train(
    lgb_params, full_data,
    num_boost_round=best_iter,
)

# Load sample_submission
sample_sub = pd.read_csv('../data/sample_submission.csv', parse_dates=['Date'])
print(f'Test period: {sample_sub["Date"].min()} → {sample_sub["Date"].max()}')

# TODO: Tạo features cho tập test tương tự tập train
# Lưu ý: lag features phải dùng giá trị từ train (không được dùng Revenue/COGS thực tế của test)

# X_test = ... (feature engineering cho test set)
# pred_revenue = final_full_model.predict(X_test)
# pred_cogs    = ... (dự báo COGS tương tự)

# submission = sample_sub.copy()
# submission['Revenue'] = pred_revenue
# submission['COGS']    = pred_cogs
# submission.to_csv('../submission/submission.csv', index=False)
# print('submission.csv saved!')
# submission.head()